<img align="left" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="right">Run machine learning models on footage</h1>
<h3 align="right"><a href="https://colab.research.google.com/github/ocean-data-factory-sweden/kso/blob/main/notebooks/publish/Publish_observations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a></h3>
<h3 align="right">Written by the KSO team</h3>

# Set up KSO requirements

### Install requirements and load KSO modules

Installing the requirements in Google Colab takes ~4 mins and might automatically crash/restart the session. Please run this cell until you get the "KSO successfully imported!" message.

In [ ]:
%matplotlib inline
import os
import sys


def initiate_dev_version():
    kso_path = os.path.abspath(os.path.join(os.getcwd(), "../.."))
    if os.path.isdir(os.path.join(kso_path, "kso_utils")):
        sys.path.insert(0, kso_path)
        %load_ext autoreload
        %autoreload 2
        print("Development mode ON - kso-utils added to the system.")
    else:
        raise FileNotFoundError("kso_utils directory not found in the expected path.")


def install_kso_utils():
    !pip install -q kso-utils
    # Temporary workaround to install panoptes from the source (avoid requests incompatibility)
    !pip install git+https://github.com/zooniverse/panoptes-python-client.git
    print("Restarting runtime to apply package changes...")
    os.kill(os.getpid(), 9)


try:
    initiate_dev_version()
    import kso_utils.widgets as kso_widgets
    import kso_utils.project_utils as p_utils
    import kso_utils.yolo_utils as y_utils
    from kso_utils.MLProjectProcessor import MLProjectProcessor

    print("KSO dev successfully imported!")
except Exception as e:
    install_kso_utils()
    import kso_utils.widgets as kso_widgets
    import kso_utils.project_utils as p_utils
    import kso_utils.yolo_utils as y_utils
    from kso_utils.MLProjectProcessor import MLProjectProcessor

    print("KSO PyPi successfully imported!")

In [ ]:
project_name = "Template project"  # available projects can be found in "../kso_utils/db_starter/projects_list.csv"
model = ""  # ???
download_dir = "."
save_dir = "."
conf_thres = 0.5  # should be between 0-1
exp_name = "Experiment"

### Initiate project's database

In [ ]:
# Find project
project = p_utils.find_project(project_name=project_name)
# Initialise mlp
mlp = MLProjectProcessor(project)

# Run model on footage

### Download and list the models from Zenodo

In [ ]:
# Show the models that are available on zenodo
mlp.get_and_show_models_zenodo(download_dir)

In [ ]:
# Get the path to the model
artifact_dir = mlp.get_model(model, download_dir)

### Choose the footage to run the models into

In [ ]:
mlp.choose_footage_source()

In [ ]:
mlp.choose_footage()

In [ ]:
# Ensure the selected footage and paths are loaded to the system
mlp.check_selected_movies()

### Run model over selected footage

In [ ]:
# Get the paths of the movies selected
mlp.detect_yolo(
    save_dir=save_dir,
    conf_thres=conf_thres,
    artifact_dir=artifact_dir,
    save_output=True,
    project=mlp.project_name,
    name=exp_name,
    model=model,
    out_format="yolo",
    source=(
        mlp.selected_movies_paths
        if isinstance(mlp.selected_movies_paths, str)
        else mlp.selected_movies_paths[0]
    ),
)

### View the processed footage

In [ ]:
kso_widgets.select_viewer()

### Process the detections
Add the metadata associated with the species identified and the movies

In [ ]:
mlp.eval_dir = (
    "/Users/jurie.germishuys/Workspace/odf/koster-uw/kso/notebooks/publish/exp_name7"
)

In [ ]:
dets_df = mlp.process_detections(
    project=mlp.project,
    db_connection=mlp.db_connection,
    csv_paths=mlp.csv_paths,
    annotations_csv_path=mlp.eval_dir,
    model_registry=mlp.registry,
    model=model,
    team_name=mlp.team_name,
    project_name=mlp.project_name,
)

In [ ]:
dets_df

### Plot the processed detections

In [ ]:
mlp.plot_processed_detections(
    df=dets_df,
    thres=10,  # number of seconds for thresholding in interval
    int_length=10,  # length in seconds of interval for filtering
)

OPTIONAL #1 - Download the processed detections in a csv file for further analysis (e.g. comparisons between citizen scientists and experts)

In [ ]:
mlp.download_detections_csv(dets_df)

OPTIONAL #2 - Processed classifications with species as columns (For biodiversity purposes)

In [ ]:
mlp.download_detections_species_cols_csv(
    df=dets_df,
)

OPTIONAL #3 - Download maxN annotations in GBIF/OBIS format (For biodiversity purposes)

In [ ]:
mlp.download_gbif_occurrences("ml_algorithms", dets_df)

OPTIONAL #4 (Required!) - Load the path of the csv files

In [ ]:
mlp.eval_dir = save_dir  # ??WHAT DOES THIS DO?

In [ ]:
# END